# PRIMEROS MODELOS PROYECTO NivELE

In [ ]:
#Cargamos el dataset
import pandas as pd
from google.colab import drive
drive.mount('/content/drive/')
path = "/content/drive/MyDrive/ColabNotebooks/APLICACIONES/ALBERTO/"

data = path + "texts_processed.csv"

#Leer el archivo
dataset = pd.read_csv(data, sep=',', quotechar='"')
dataset.head()

Mounted at /content/drive/


,id,label,text
0,1,A1,Mientras caminaba por las carreteras mientras ...
1,2,A1,"Cuando Chaplin estaba caminando, vio a un niño..."
2,3,A2,"Cuando Chaplin caminando en la calle, encotró ..."
3,4,A2,En el vídeo Charles Chaplin en el calle ve un ...
4,5,A2,Cuando Chaplin está pasando y fumar contra un ...


In [ ]:
dataset.shape
print(f"Número de filas: {dataset.shape[0]}")
print(f"Número de columnas: {dataset.shape[1]}")

Número de filas: 3556
Número de columnas: 3


In [ ]:
class_counts = dataset["label"].value_counts()
class_counts

,count
label,
C1,1094
B2,969
B1,763
A2,614
A1,116


In [ ]:
print("Valores NA en sampled_dataset:")
display(dataset.isnull().sum())  #No hay valores NA

Valores NA en sampled_dataset:


,0
id,0
label,0
text,0


In [ ]:
print("Ejemplo de texto A2:\n")
print(dataset[dataset["label"] == "A2"]["text"].iloc[0])

Ejemplo de texto A2:

Cuando Chaplin caminando en la calle, encotró un bebè . Trato de encontrar a su el padre, Pero èl no los encontro. Así que tartó de dar el bebè a Cualquier persona en la calle , todos sus intentos fueron infructuosos. La primera, segunda y tercera vez, y despuès del cuarto intento , renunció y aceptó la adopción del niño.Finalmente, Chaplin decidió mantener al bebè, porque de la carta en los paños del bebè.


In [ ]:
print("Ejemplo de texto C1:\n")
print(dataset[dataset["label"] == "C1"]["text"].iloc[0])

Ejemplo de texto C1:

El vídeo presenta un hombre que se llama Chaplin estaba andando sacó el cigarrillo y lo encendió en la calle, parece que encontró un niño tirado en el suelo, nadie aceptó al niño, se escapó del policía, por lo que se lo llevó a una de las mujeres de la calle, pero ella no lo aceptó y lo golpeó.. intentó de buscar su familia, finalmente no ha encontrado a nadie, encontró un trozo de papel dentro de la ropa del niño con algo escrito y luego lo dejó con él.


# **MODELO Bag of words**

In [ ]:
#Seleccionamos los datos de X, features
X = dataset["text"].values
y = dataset["label"].values

In [ ]:
#Separamos los datos:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from keras.utils import to_categorical

# Lo he separado en dos
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Encode labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)

# One-hot encoding
num_clases = len(label_encoder.classes_)
y_train = to_categorical(y_train_encoded, num_classes=num_clases)
y_val = to_categorical(y_val_encoded, num_classes=num_clases)

print("Tamaños")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")
print(f"Número de clases: {num_clases}")

Tamaños
X_train: (2844,), y_train: (2844, 5)
X_val: (712,), y_val: (712, 5)
Número de clases: 5


In [ ]:
#Creamos un modelo bag of words
from keras import layers
import numpy as np
import keras

max_tokens = 20000

text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    ngrams=2,
    output_mode="tf_idf",
)


text_vectorization.adapt(X_train)

X_train_bow = text_vectorization(X_train)
X_val_bow = text_vectorization(X_val)


In [ ]:
#Compilamos
def build_linear_classifier(max_tokens, num_clases):
    inputs = keras.Input(shape=(max_tokens,))
    x = layers.Dense(64, activation="relu")(inputs)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_clases, activation="softmax")(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

model_bow = build_linear_classifier(max_tokens, num_clases)

In [ ]:
#Y ajustamos
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    restore_best_weights=True,
    patience=3,
)

history = model_bow.fit(
    X_train_bow, y_train,
    validation_data= (X_val_bow, y_val),
    epochs=20,
    callbacks=[early_stopping]
)


Epoch 1/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.3871 - loss: 1.7515 - val_accuracy: 0.4958 - val_loss: 1.2140
Epoch 2/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.5823 - loss: 1.0143 - val_accuracy: 0.5126 - val_loss: 1.1392
Epoch 3/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.7120 - loss: 0.7559 - val_accuracy: 0.5042 - val_loss: 1.1925
Epoch 4/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step - accuracy: 0.7792 - loss: 0.5600 - val_accuracy: 0.5070 - val_loss: 1.3031
Epoch 5/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.8428 - loss: 0.4373 - val_accuracy: 0.5042 - val_loss: 1.3531


In [ ]:
model_bow.summary(line_length=80)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                      ┃ Output Shape             ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)          │ (None, 20000)            │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dense (Dense)                     │ (None, 64)               │     1,280,064 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dropout (Dropout)                 │ (None, 64)               │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dense_1 (Dense)                   │ (None, 5)                │           325 │
└───────────────────────────────────┴──────────────────────────┴───────────────┘

 Total params: 3,841,169 (14.65 MB)

 Trainable params: 1,280,389 (4.88 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,560,780 (9.77 MB)

# **MODELO EMBEDDINGS**

In [ ]:
import tensorflow as tf
import keras
from tensorflow.keras.layers import TextVectorization

max_length = 200
max_tokens = 20000

text_vectorization = TextVectorization(
    max_tokens=max_tokens,
    split="whitespace",
    output_mode="int",
    output_sequence_length=max_length,
)

text_vectorization.adapt(X_train)


batch_size = 32
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))

def map_function(x, y):
    vec_x = text_vectorization(x)

    vec_x_padded_and_truncated = tf.concat(
        [vec_x, tf.zeros(max_length, dtype=tf.int64)], axis=0
    )[:max_length]
    vec_x_padded_and_truncated = tf.ensure_shape(vec_x_padded_and_truncated, (max_length,))

    return vec_x_padded_and_truncated, y

sequence_train_ds = train_ds.map(
    map_function, num_parallel_calls=tf.data.AUTOTUNE
).batch(batch_size).prefetch(tf.data.AUTOTUNE)

sequence_val_ds = val_ds.map(
    map_function, num_parallel_calls=tf.data.AUTOTUNE
).batch(batch_size).prefetch(tf.data.AUTOTUNE)


In [ ]:
#Creamos un modelo con wordEmbedding
inputs = keras.Input(shape=(max_length,))
x = layers.Embedding(max_tokens, 128)(inputs)
x = layers.Bidirectional(layers.LSTM(64))(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(num_clases, activation="softmax")(x)

model_embed = keras.Model(inputs, outputs)

In [ ]:
#Compilamos
model_embed.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)


In [ ]:
model_embed.summary(line_length=80)

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                      ┃ Output Shape             ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)        │ (None, 200)              │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ embedding_4 (Embedding)           │ (None, 200, 128)         │     2,560,000 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional)   │ (None, 128)              │        98,816 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dense_7 (Dense)                   │ (None, 256)              │        33,024 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dropout_4 (Dropout)               │ (None, 256)              │             0 │
├───────────────────────────────────┼──────────────────────────┼───────────────┤
│ dense_8 (Dense)                   │ (None, 5)                │         1,285 │
└───────────────────────────────────┴──────────────────────────┴───────────────┘

 Total params: 2,693,125 (10.27 MB)

 Trainable params: 2,693,125 (10.27 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    restore_best_weights=True,
    patience=2,
)

history_embed = model_embed.fit(
    sequence_train_ds,
    validation_data=sequence_val_ds,
    epochs=20,
    callbacks=[early_stopping]
)

Epoch 1/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 35s 344ms/step - accuracy: 0.3998 - loss: 1.3163 - val_accuracy: 0.4874 - val_loss: 1.1743
Epoch 2/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 38s 315ms/step - accuracy: 0.5461 - loss: 1.0365 - val_accuracy: 0.5042 - val_loss: 1.1850
Epoch 3/20
89/89 ━━━━━━━━━━━━━━━━━━━━ 41s 307ms/step - accuracy: 0.6523 - loss: 0.8458 - val_accuracy: 0.4649 - val_loss: 1.3302


Veamos los ejemplos mal clasificados.

In [ ]:
#Vamos a buscar los ejemplos que fallan y reflexionaremos sobre por qué lo hacen.
y_pred_prob = model_bow.predict(X_val_bow)
print(y_pred_prob)

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
[[0.00341141 0.09278114 0.55179554 0.19696407 0.1550477 ]
 [0.09943831 0.14660053 0.19220813 0.34237    0.21938303]
 [0.22255845 0.45056665 0.12652358 0.0833714  0.11698005]
 ...
 [0.20856889 0.4246073  0.16089128 0.0841392  0.12179338]
 [0.00928839 0.02224878 0.06302907 0.3124188  0.59301496]
 [0.00837813 0.01894562 0.03410722 0.35338634 0.58518267]]


In [ ]:
# Obtenemos las etiquetas predichas a partir de las probabilidades
# La clase predicha es aquella con la mayor probabilidad
y_pred_labels = np.argmax(y_pred_prob, axis=1)
y_pred_labels

array([2, 3, 1, 4, 2, 1, 1, 3, 3, 1, 3, 2, 3, 4, 1, 3, 3, 1, 2, 3, 3, 3,
       3, 3, 4, 4, 4, 3, 2, 4, 4, 3, 4, 3, 3, 3, 1, 1, 2, 2, 4, 4, 4, 4,
       3, 3, 1, 3, 4, 4, 1, 3, 3, 3, 1, 3, 2, 4, 3, 4, 2, 4, 3, 4, 4, 2,
       3, 4, 2, 2, 4, 4, 4, 2, 3, 2, 2, 2, 4, 4, 3, 1, 3, 1, 4, 4, 2, 2,
       4, 1, 3, 3, 3, 2, 1, 1, 3, 4, 1, 4, 2, 4, 3, 3, 3, 4, 2, 4, 4, 3,
       1, 4, 3, 3, 4, 3, 3, 4, 2, 2, 3, 3, 3, 3, 3, 1, 3, 3, 1, 4, 3, 2,
       2, 3, 2, 4, 2, 3, 1, 4, 3, 2, 3, 4, 3, 2, 4, 2, 4, 3, 3, 1, 3, 4,
       1, 3, 4, 2, 2, 1, 1, 3, 2, 2, 4, 4, 4, 3, 4, 3, 4, 4, 4, 3, 2, 4,
       4, 1, 4, 3, 3, 4, 1, 3, 1, 1, 2, 4, 3, 4, 3, 4, 4, 2, 4, 2, 1, 4,
       4, 4, 4, 4, 3, 3, 1, 1, 2, 3, 3, 4, 4, 4, 3, 2, 4, 3, 4, 3, 2, 2,
       1, 4, 4, 2, 4, 3, 3, 4, 4, 3, 3, 2, 1, 2, 3, 4, 1, 4, 4, 3, 4, 4,
       3, 4, 4, 3, 2, 3, 2, 4, 4, 4, 2, 3, 1, 4, 2, 3, 3, 4, 3, 4, 1, 4,
       4, 1, 1, 2, 2, 3, 4, 4, 4, 2, 4, 1, 3, 1, 3, 3, 2, 4, 1, 4, 4, 1,
       4, 1, 3, 4, 2, 4, 1, 2, 4, 3, 3, 2, 4, 2, 3,

In [ ]:
#Buscamos los índices de las instancias con predicciones incorrectas
import numpy as np
# Convertir y_val de one-hot encoding a etiquetas de clase para la comparación
y_val_classes = np.argmax(y_val, axis=1)
indices_incorrectos = np.where(y_pred_labels != y_val_classes)[0]
indices_incorrectos

array([  5,   6,   8,   9,  10,  11,  14,  16,  19,  21,  22,  27,  28,
        31,  33,  34,  36,  39,  41,  43,  45,  46,  47,  49,  52,  54,
        55,  56,  58,  60,  62,  64,  65,  68,  69,  72,  73,  74,  75,
        76,  77,  78,  82,  88,  89,  91,  94,  96, 100, 101, 104, 105,
       106, 111, 112, 116, 117, 118, 119, 120, 121, 122, 124, 126, 129,
       132, 133, 134, 136, 137, 138, 140, 141, 142, 143, 145, 153, 155,
       156, 157, 160, 163, 164, 172, 173, 175, 177, 178, 181, 182, 183,
       185, 186, 188, 190, 194, 195, 197, 198, 200, 201, 202, 204, 205,
       207, 209, 210, 213, 215, 216, 221, 222, 223, 225, 232, 234, 236,
       239, 242, 246, 247, 248, 250, 252, 253, 254, 255, 256, 260, 261,
       262, 266, 267, 268, 270, 272, 274, 278, 279, 280, 282, 285, 287,
       298, 299, 300, 302, 303, 304, 305, 306, 308, 309, 310, 312, 313,
       314, 316, 320, 321, 322, 324, 327, 328, 330, 332, 333, 335, 337,
       338, 341, 342, 343, 344, 345, 349, 351, 357, 358, 360, 36

In [ ]:
#Mostramos las 10 primeras instancias con predicciones incorrectas
for i in indices_incorrectos[:10]:
    print(f"Instancia {i}: Predicción={y_pred_labels[i]}, Real={y_val[i]}")

Instancia 5: Predicción=1, Real=[0. 0. 1. 0. 0.]
Instancia 6: Predicción=1, Real=[0. 0. 1. 0. 0.]
Instancia 8: Predicción=3, Real=[0. 0. 0. 0. 1.]
Instancia 9: Predicción=1, Real=[0. 0. 1. 0. 0.]
Instancia 10: Predicción=3, Real=[0. 0. 1. 0. 0.]
Instancia 11: Predicción=2, Real=[0. 1. 0. 0. 0.]
Instancia 14: Predicción=1, Real=[0. 0. 1. 0. 0.]
Instancia 16: Predicción=3, Real=[0. 0. 0. 0. 1.]
Instancia 19: Predicción=3, Real=[0. 0. 1. 0. 0.]
Instancia 21: Predicción=3, Real=[0. 0. 1. 0. 0.]


In [ ]:
#Volvemos al texto principal antes de la vectorización para buscar respuestas
for i in indices_incorrectos[:10]:
    print(f"\nInstancia {i}:")
    print("  Texto:", X_val[i])
    print("  Predicción:", y_pred_labels[i])
    print("  Real:", y_val[i])


Instancia 5:
  Texto: Soy de Houston, Texas. Houston es muy caliente in el verano pero el tiempo es muy bien en el otros estaciones. Hay muchas hacer en Houston. Estoy un joven, y mis amigos yo van a Austin, Texas a el Universidad del Texas. Tenemos otros amigos en la escuela, y nosotros visitamos ellos mucho. Hay un centro commercial muy bien en Houston y la ciudad es muy grande. Hay muchas casas bonitas cerca de mi casa, y parques bonitas. Mis amigos y yo queremos ir de compras durante el fin de semana. Luego, hay muchas restaurantes muy deliciosas, y nosotros llevamos en ropa bonita y vamos a los restaurantes. Houston es muy diversidad y tengo muchas culturas en mi escuela. Hay muchas personas indigenes de Mexico en Texas. En el invierno, hay un rodeo muy grande y muchas personas attenden. Hay muchas deportes en Houston. Hay futbol americano, futbol, tenis, y otras deportes. Mis amigos y yo nos gusta jugar tenis en mi club. Tambien, mi familia le gusta jugar tenis con otras. Ahora,